 # Preprocessing 2: normalization & lemmatization

Pipeline:

1) Normalization
    - Lower case
    - Check unicode + removing unsupported characters
    - Sentence splitting
    - Handling "borken" words (e.g., words with "-" inside)
    - Double spaces normalization
    - Punctuation normalization 


2) Text preparation  for FastTextIt
    - Tokenization
    - Punctuation removal


In [ ]:
import re
import unicodedata
from pathlib import Path   
from typing import List
import spacy
import os

# carica il modello una sola volta (fuori dalla funzione)
nlp = spacy.load("it_core_news_sm", disable=["parser", "ner"])

1) Normalization & lemmatization

In [ ]:

def pulisci_testo(testo: str) -> str:
  

    # normalizzazione Unicode (accenti, caratteri strani)
    testo = unicodedata.normalize("NFKC", testo)

    # lowercase      
    testo = testo.lower()
    
    # normalizzazione apostrofi e virgolette
    testo = testo.replace("’", "'").replace("‘", "'")
    testo = testo.replace("“", '"').replace("”", '"')

    # rimozione parole spezzate da trattino a capo
    # tipo se abbiamo: "pa-\nrola" → "parola"
    testo = re.sub(r"-\s*\n\s*", "", testo)

    # rimozione caratteri non supportati
    testo = re.sub(r"[^a-zàèéìòù0-9.,;:!?'\-\s\"]", " ", testo)

    # normalizzazione spazi multipli
    testo = re.sub(r"\s+", " ", testo)

    # split in frasi (una per riga)
    frasi = re.split(r"(?<=[.!?])\s+", testo)

    # faccio qui lemmatizzazione
    frasi_lemmatizzate = []

    for frase in frasi:
        doc = nlp(frase)
        lemmi = [
            token.lemma_
            for token in doc
            if not token.is_space
        ]
        frasi_lemmatizzate.append(" ".join(lemmi))

    testo_finale = "\n".join(frasi_lemmatizzate)

    return testo_finale.strip()

2) Tokenization & Punctuation removal

In [3]:
def tokenizza_e_rimuovi_punteggiatura(testo: str) -> List[str]:
    # sostituiamo la punteggiatura con spazio (apostrofi OK)
    testo = re.sub(r"[.,;:!?\"()\[\]{}<>]", " ", testo)

    # normalizziamo di nuovo gli spazi
    testo = re.sub(r"\s+", " ", testo).strip()

    # tokenizziamo
    tokens = testo.split(" ")

    return tokens

In [4]:
def testo_per_fasttext(testo: str) -> str:
    """
    Restituisce il testo pronto per FastText:
    - tokenizza ogni frase
    - una frase per riga
    """
    frasi = testo.split("\n")
    righe_tokenizzate = []

    for frase in frasi:
        tokens = tokenizza_e_rimuovi_punteggiatura(frase)
        if tokens:
            righe_tokenizzate.append(" ".join(tokens))

    return "\n".join(righe_tokenizzate)


In [ ]:
for file in os.listdir("../year_clean/"):
    
    print(f"File: {file} ", end="")
    
    with open(f"../ocr_clean/{file}", "r", encoding="utf-8") as f:
        testo = f.read()

    testo_pulito = pulisci_testo(testo)
    testo_fasttext = testo_per_fasttext(testo_pulito)

    filename_without_extension = Path(file).stem
    
    with open(f"../lemmas/{filename_without_extension}_lemmas.txt", "w", encoding="utf-8") as f:
        f.write(testo_fasttext)

print("OK")

OK


Removing proper nouns

In [ ]:
import spacy
import unicodedata
import re
import os
from pathlib import Path

# --- CONFIGURAZIONE SPACY ---
try:
    # Carichiamo modello disabilitando parser e ner per velocità e memoria
    nlp = spacy.load("it_core_news_lg", disable=["parser", "ner"])
    # Aggiungiamo sentencizer per dividere le frasi senza il parser pesante
    if "sentencizer" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")
    nlp.max_length = 2000000 
except Exception as e:
    print(f"Errore modello: {e}. Esegui: python -m spacy download it_core_news_lg")
    exit()

def pulizia_preliminare_stringa(testo: str) -> str:
    """
    Replica la pulizia iniziale MA mantiene le maiuscole
    per permettere a Spacy di riconoscere i nomi propri.
    """
    # 1. Normalizzazione Unicode
    testo = unicodedata.normalize("NFKC", testo)
    
    # 2. Normalizzazione virgolette (come nel tuo codice originale)
    testo = testo.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    
    # 3. Rimozione parole spezzate da trattino a capo (TUA LOGICA)
    testo = re.sub(r"-\s*\n\s*", "", testo)
    
    # 4. Normalizzazione spazi
    testo = re.sub(r"\s+", " ", testo)
    
    return testo

def processa_chunk_spacy(doc):
    """Prende un doc Spacy e restituisce le stringhe pronte per FastText"""
    frasi_output = []
    
    for sent in doc.sents:
        tokens_validi = []
        for token in sent:
            # --- FILTRI ---
            
            # 1. Via i Nomi Propri 
            if token.pos_ == "PROPN":
                continue
                
            # 2. Via Punteggiatura e Numeri
            if token.is_punct or token.is_space or token.like_num:
                continue

            # --- LEMMATIZZAZIONE ---
            lemma = token.lemma_.lower()
            
            # 4. Pulizia finale del lemma (Simulazione del tuo regex [^a-z])
            # Accettiamo il lemma solo se contiene caratteri validi
            # Questo rimuove residui strani che il vecchio regex toglieva
            if not re.match(r'^[a-zàèéìòù0-9\-\']+$', lemma):
                continue

            tokens_validi.append(lemma)
            
        if tokens_validi:
            frasi_output.append(" ".join(tokens_validi))
            
    return "\n".join(frasi_output)

def generator_lettura_file(file_path, chunk_size=100000):
    """Legge il file a pezzetti per non saturare la RAM"""
    with open(file_path, "r", encoding="utf-8") as f:
        while True:
            data = f.read(chunk_size)
            if not data:
                break
            yield data

# --- ESECUZIONE ---

input_dir = Path("../ocr_clean/") # La tua cartella input
output_dir = Path("../lemmas/lemmas_no_propn/")
output_dir.mkdir(parents=True, exist_ok=True)

print("Inizio pipeline ottimizzata (Streaming + No PROPN)...")

for file_name in os.listdir(input_dir):
    if not file_name.endswith(".txt"): 
        continue
        
    print(f"Processing: {file_name}...", end="")
    
    file_path = input_dir / file_name
    out_path = output_dir / f"{Path(file_name).stem}_lemmas_no_propn.txt"
    
    try:
        # Creiamo/Puliamo il file di output
        with open(out_path, "w", encoding="utf-8") as f_out:
            
            # Creiamo il generatore di testo
            stream_testo = generator_lettura_file(file_path)
            
            # Applichiamo la pulizia preliminare al volo su ogni chunk
            stream_pulito = (pulizia_preliminare_stringa(chunk) for chunk in stream_testo)
            
            # Spacy processa lo stream
            # n_process=1 (sicuro), batch_size ridotto per memoria
            for doc in nlp.pipe(stream_pulito, batch_size=20):
                
                txt_processato = processa_chunk_spacy(doc)
                
                if txt_processato:
                    f_out.write(txt_processato + "\n")
                    
        print(" OK.")

    except Exception as e:
        print(f" ERRORE: {e}")

print("Tutti i file elaborati.")

Inizio pipeline ottimizzata (Streaming + No PROPN)...
Processing: un_1948_clean.txt... OK.
Processing: un_1949_clean.txt... OK.
Processing: un_1950_clean.txt... OK.
Processing: un_1951_clean.txt... OK.
Processing: un_1952_clean.txt... OK.
Processing: un_1953_clean.txt... OK.
Processing: un_1954_clean.txt... OK.
Processing: un_1955_clean.txt... OK.
Processing: un_1956_clean.txt... OK.
Processing: un_1957_clean.txt... OK.
Processing: un_1958_clean.txt... OK.
Processing: un_1959_clean.txt... OK.
Processing: un_1960_clean.txt... OK.
Processing: un_1961_clean.txt... OK.
Processing: un_1962_clean.txt... OK.
Processing: un_1963_clean.txt... OK.
Processing: un_1964_clean.txt... OK.
Processing: un_1965_clean.txt... OK.
Processing: un_1966_clean.txt... OK.
Processing: un_1967_clean.txt... OK.
Processing: un_1968_clean.txt... OK.
Processing: un_1969_clean.txt... OK.
Processing: un_1970_clean.txt... OK.
Processing: un_1971_clean.txt... OK.
Processing: un_1972_clean.txt... OK.
Processing: un_1973_c